In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv("../train.csv")

X = df.drop(columns=["Price_INR_Lakhs", "Property_ID"])
y = df["Price_INR_Lakhs"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=42
)

categorical_features = [
    "City", "Locality_Type", "Property_Type", "Furnishing_Status"
]

numerical_features = [
    "BHK", "Bathrooms", "Super_Area_SqFt", "Carpet_Area_SqFt",
    "Floor_Number", "Total_Floors", "Age_of_Property",
    "Parking", "Lift_Available", "Gated_Community",
    "Distance_to_Metro_km", "Distance_to_City_Center_km"
]

numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)],
    sparse_threshold=0
)

### 5-fold cross-validation: baseline vs tuned

The tuned model only beat the baseline by about 0.004 R2 on one validation split.
Cross-validation on the full training data tells us if that gain is consistent.

In [2]:
from sklearn.model_selection import KFold, cross_validate

baseline_params = {
    "learning_rate": 0.10,
    "max_iter": 100,
    "max_leaf_nodes": 31
}

tuned_params = {
    "learning_rate": 0.05,
    "max_iter": 500,
    "max_leaf_nodes": 15,
    "min_samples_leaf": 100,
    "l2_regularization": 1.0
}

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

cv_results = {}

for name, params in [("baseline", baseline_params), ("tuned", tuned_params)]:
    model = Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", HistGradientBoostingRegressor(random_state=42, **params))
    ])

    scores = cross_validate(
        model, X, y, cv=kfold,
        scoring=["r2", "neg_mean_absolute_error"],
        n_jobs=-1
    )

    cv_results[name] = scores
    print(f"{name:<9} R2 per fold: {scores['test_r2'].round(4)}")

baseline  R2 per fold: [0.7849 0.7801 0.7819 0.7913 0.783 ]


tuned     R2 per fold: [0.7889 0.7866 0.783  0.795  0.7864]


In [3]:
summary = pd.DataFrame({
    name: {
        "R2_mean": scores["test_r2"].mean(),
        "R2_std": scores["test_r2"].std(),
        "MAE_mean": -scores["test_neg_mean_absolute_error"].mean()
    }
    for name, scores in cv_results.items()
}).T

summary

,R2_mean,R2_std,MAE_mean
baseline,0.784217,0.003854,32.996578
tuned,0.787966,0.003996,32.758237


The tuned model wins on all 5 folds (mean R2 0.7880 vs 0.7842, MAE about 0.24 lakhs lower), so the improvement is real, if small.